# OMSCGR — Índice de brecha de videovigilancia por hexágono (DMQ, 2025)

Pipeline reproducible: construye el **índice compuesto de intensidad de inseguridad**, la **métrica de brecha**, la **asignación de 600 puntos de conexión** (3 cámaras físicas cada uno) por cuota zonal, el **cruce con espacio público y equipamientos institucionales**, la **validación por correlación**, la **validación externa** contra el Índice de Convivencia Ciudadana (ICC) y la **brecha remanente** por hexágono.

**Nota sobre el periodo:** todas las cifras de HI, ROBOS y LIBADOR son el acumulado de los doce meses de 2025 por hexágono, no un corte puntual ni una tendencia reciente.

**Entradas esperadas** (mismo `OBJECTID` entre ambas):
- `260716_Hexagonos_2025_incidentes_infraestructura_DMQ_v3.xlsx` — atributos por hexágono (ver diccionario de variables v2, incluye PARQUE_15/Nm_15PARQ; ya no trae GEOCERCA/QUEBRADA/P_DESNIV)
- `hex_300m_incidentes_v2_AZ.geojson` — misma información + geometría (solo necesaria para el mapa)

**Autor:** OMSCGR / Secretaría General de Seguridad Ciudadana y Gestión de Riesgos


In [1]:
import json
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 160)


## 0. Configuración

In [2]:
RUTA_XLSX = "260716_Hexagonos_2025_incidentes_infraestructura_DMQ_v3.xlsx"
HOJA = "Hoja1"
N_PUNTOS_NUEVOS = 600           # puntos de conexión nuevos a distribuir
CAMARAS_POR_PUNTO = 3           # cada punto de conexión instala 3 cámaras físicas
COMPONENTES_INDICE = ["HI", "ROBOS", "LIBADOR"]  # peso igual, ver informe §3.1

# Fuentes de cámaras YA instaladas. Son 2 categorías, confirmado por el
# diccionario de variables v2 y por la propia base v3 (ya no incluye
# GEOCERCA, QUEBRADA ni P_DESNIV): EP EMSEGURIDAD y Ecu911.
FUENTES_CAMARA_ACTUAL = ["CAM_EP", "CAM_ECU"]


## 1. Carga de datos

In [3]:
def cargar_base(ruta=RUTA_XLSX, hoja=HOJA):
    df = pd.read_excel(ruta, sheet_name=hoja)
    if "OBJECTID_1" in df.columns and "OBJECTID" not in df.columns:
        df = df.rename(columns={"OBJECTID_1": "OBJECTID"})
    df["N_CAM"] = df[FUENTES_CAMARA_ACTUAL].sum(axis=1)
    df["ANY_CAM"] = (df["N_CAM"] > 0).astype(int)
    return df

df = cargar_base()
print("Hexágonos:", len(df), "| columnas:", len(df.columns))
df.head(3)


Hexágonos: 18640 | columnas: 34


,OBJECTID,Input_FID,dpa_barrio,Nombre_Bar,Cod_Barrio,Parroquia,Cod_Parroq,Admin_Zona,AÑO,LIBADOR,ESCAN,VCSSF,ROBOS,HI,CAM_EP,...,P_BUS,P_METRO,SENDERO,UPC,PMO_CH,PMO_MARIS,UNIVERSID,POLIG_ALT,ICC,ZONA_0,NZONA_0,PARQUE_15,Nm_15PARQ,N_CAM,ANY_CAM
0,1,16321,170166001,CHIRIBOGA,EA_170166001,LLOA,170166,ELOY ALFARO,2025,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,NaN,0,0
1,2,16322,170166001,CHIRIBOGA,EA_170166001,LLOA,170166,ELOY ALFARO,2025,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,NaN,0,0
2,3,15724,170166001,CHIRIBOGA,EA_170166001,LLOA,170166,ELOY ALFARO,2025,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,NaN,0,0


## 2. Índice compuesto de intensidad de inseguridad (§3.1 del informe)

Para cada hexágono se calcula el puntaje estandarizado (z-score) de HI, ROBOS y LIBADOR sobre el universo de 18.640 unidades, y se promedian con peso igual:

$$z(x) = \\frac{x - \\mu_x}{\\sigma_x}$$

donde μₓ y σₓ son la media y la desviación estándar de la variable x. La estandarización es necesaria porque los tres fenómenos operan en escalas muy distintas (un hexágono puede tener centenares de libadores pero pocos homicidios); sin ella, la variable de mayor magnitud absoluta dominaría el índice por artefacto de escala.

$$\\text{ÍNDICE} = \\text{promedio}(z(HI), z(ROBOS), z(LIBADOR))$$

reescalado a 0–100 por min-máx para su lectura institucional.


In [4]:
def calcular_indice(df, componentes=COMPONENTES_INDICE):
    """z-score de cada componente (peso igual) -> promedio -> reescalado 0-100."""
    for c in componentes:
        mu, sd = df[c].mean(), df[c].std()
        df[f"{c}_z"] = (df[c] - mu) / sd

    df["INDICE_z"] = df[[f"{c}_z" for c in componentes]].mean(axis=1)

    imin, imax = df["INDICE_z"].min(), df["INDICE_z"].max()
    df["INDICE_100"] = (df["INDICE_z"] - imin) / (imax - imin) * 100
    return df

df = calcular_indice(df)

for c in COMPONENTES_INDICE:
    print(f"{c}: media={df[c].mean():.4f}  desv.est={df[c].std():.4f}")

print()
print("Ejemplo — mismo valor absoluto (1), z distinto según el fenómeno:")
mu_hi, sd_hi = df['HI'].mean(), df['HI'].std()
mu_ro, sd_ro = df['ROBOS'].mean(), df['ROBOS'].std()
print(f"  z(HI=1)    = ({1}-{mu_hi:.4f})/{sd_hi:.4f} = {(1-mu_hi)/sd_hi:.2f}")
print(f"  z(ROBOS=1) = ({1}-{mu_ro:.4f})/{sd_ro:.4f} = {(1-mu_ro)/sd_ro:.2f}")


HI: media=0.0145  desv.est=0.1820
ROBOS: media=0.8071  desv.est=4.6412
LIBADOR: media=2.2788  desv.est=17.9083

Ejemplo — mismo valor absoluto (1), z distinto según el fenómeno:
  z(HI=1)    = (1-0.0145)/0.1820 = 5.42
  z(ROBOS=1) = (1-0.8071)/4.6412 = 0.04


## 3. Brecha: preservar la infraestructura instalada (§3.2 del informe)

$$\\text{BRECHA} = \\text{ÍNDICE} \\times \\frac{1}{1 + \\text{cámaras actuales}}$$

Un hexágono sin ninguna cámara conserva el 100% de su índice como brecha; uno con una cámara, la mitad; con dos, un tercio. Esta forma funcional no remueve ni reubica ninguna cámara existente — cumple la restricción por diseño, no por corrección posterior.


In [5]:
def calcular_brecha(df):
    """BRECHA = ÍNDICE × [1 / (1 + cámaras actuales)]"""
    df["FACTOR_COBERTURA"] = 1 / (1 + df["N_CAM"])
    df["BRECHA"] = df["INDICE_100"] * df["FACTOR_COBERTURA"]
    return df

df = calcular_brecha(df)
(df[["OBJECTID", "Nombre_Bar", "Admin_Zona", "HI", "ROBOS", "LIBADOR", "N_CAM", "INDICE_100", "BRECHA"]]
    .sort_values("BRECHA", ascending=False).head(10))


,OBJECTID,Nombre_Bar,Admin_Zona,HI,ROBOS,LIBADOR,N_CAM,INDICE_100,BRECHA
5400,5401,LA FLORESTA (MARISCAL SUCRE),LA MARISCAL,0,67,734,0,73.935133,73.935133
5446,5447,LA FLORESTA (MARISCAL SUCRE),LA MARISCAL,0,49,470,0,49.095438,49.095438
4254,4255,CLEMENTE SÁNCHEZ DE ORELLANA,QUITUMBE,4,25,119,0,45.377689,45.377689
14403,14404,SIERRA HERMOSA,CALDERÓN,0,4,533,0,40.854073,40.854073
13990,13991,LA FLORESTA (CARCELÉN),LA DELICIA,4,17,74,0,39.726095,39.726095
4237,4238,PUEBLO SOLO PUEBLO,QUITUMBE,3,22,43,0,31.522151,31.522151
4063,4064,CIUDADELA METROPOLITANA CAUPICHO,QUITUMBE,3,19,53,0,31.404782,31.404782
5482,5483,PARQUE LA CAROLINA,EUGENIO ESPEJO,0,89,60,0,30.050851,30.050851
5508,5509,BORJA YEROVI,EUGENIO ESPEJO,1,51,73,0,27.428727,27.428727
4968,4969,BARRIO UNIÓN Y JUSTICIA,ELOY ALFARO,3,10,379,1,53.102370,26.551185


## 4. Cuota por zona y ordenamiento intrazonal (§3.3 del informe)

Los 600 puntos de conexión se distribuyen en proporción al peso de cada zona sobre la brecha total del Distrito:

$$\\text{Cuota}_{zona} = 600 \\times \\frac{\\text{brecha}_{zona}}{\\text{brecha}_{total}}$$

Dentro de cada zona, la cuota se llena con los hexágonos de mayor brecha hasta agotarla.


In [6]:
def asignar_puntos_conexion(df, n_puntos=N_PUNTOS_NUEVOS):
    """Distribuye n_puntos (puntos de conexión; cada uno instala
    CAMARAS_POR_PUNTO=3 cámaras físicas) en proporción a la brecha de cada
    zona administrativa, y dentro de cada zona toma los hexágonos de mayor
    brecha hasta agotar la cuota."""
    brecha_zona = df.groupby("Admin_Zona")["BRECHA"].sum().sort_values(ascending=False)
    pct = brecha_zona / brecha_zona.sum() * 100
    cuota = (pct / 100 * n_puntos).round().astype(int)

    diff = n_puntos - cuota.sum()
    if diff != 0:
        cuota.iloc[0] += diff

    asignaciones = []
    for zona, q in cuota.items():
        candidatos = df[df["Admin_Zona"] == zona].sort_values("BRECHA", ascending=False).head(q).copy()
        candidatos["CUOTA_ZONA"] = q
        asignaciones.append(candidatos)

    asignacion = pd.concat(asignaciones).sort_values(["Admin_Zona", "BRECHA"], ascending=[True, False])
    return asignacion, cuota, pct

asignacion, cuota, pct = asignar_puntos_conexion(df)
print(f"Total puntos asignados: {len(asignacion)}")
pd.DataFrame({"% brecha": pct.round(1), "cuota (puntos)": cuota}).sort_values("cuota (puntos)", ascending=False)


Total puntos asignados: 600


,% brecha,cuota (puntos)
Admin_Zona,,
EUGENIO ESPEJO,18.9,113
LA DELICIA,16.0,96
ELOY ALFARO,14.2,85
QUITUMBE,13.8,83
TUMBACO,8.3,50
CALDERÓN,8.3,50
MANUELA SÁENZ,6.9,41
LOS CHILLOS,6.7,40
LA MARISCAL,6.0,36


## 5. Validación: correlación índice ↔ infraestructura, antes/después (§4.1)

In [7]:
def validar_correlacion(df, asignacion):
    asignados_ids = set(asignacion["OBJECTID"])
    df["N_CAM_POST"] = df["N_CAM"] + df["OBJECTID"].isin(asignados_ids).astype(int)
    df["ANY_CAM_POST"] = (df["N_CAM_POST"] > 0).astype(int)

    filas = [
        ("Correlación con n.º de cámaras (Pearson)",
         df["INDICE_100"].corr(df["N_CAM"]), df["INDICE_100"].corr(df["N_CAM_POST"])),
        ("Correlación con presencia de cámara (Pearson)",
         df["INDICE_100"].corr(df["ANY_CAM"]), df["INDICE_100"].corr(df["ANY_CAM_POST"])),
        ("Correlación por rangos (Spearman)",
         df["INDICE_100"].corr(df["N_CAM"], method="spearman"),
         df["INDICE_100"].corr(df["N_CAM_POST"], method="spearman")),
    ]
    return pd.DataFrame(filas, columns=["Medida", "Antes", "Después"]).round(3)

validacion = validar_correlacion(df, asignacion)
validacion


,Medida,Antes,Después
0,Correlación con n.º de cámaras (Pearson),0.534,0.631
1,Correlación con presencia de cámara (Pearson),0.520,0.672
2,Correlación por rangos (Spearman),0.424,0.649


## 5b. Brecha remanente por hexágono (§4.7 del informe)

Recalcula la brecha de cada hexágono asumiendo ya instalados los 600 puntos de conexión (3 cámaras físicas cada uno). Es la misma fórmula de la sección 3, aplicada sobre el inventario **post-intervención**. Sirve para priorizar una eventual segunda fase.


In [8]:
def calcular_brecha_remanente(df, asignacion):
    asignados_ids = set(asignacion["OBJECTID"])
    n_cam_post_fisico = df["N_CAM"] + df["OBJECTID"].isin(asignados_ids).astype(int) * CAMARAS_POR_PUNTO
    factor_post = 1 / (1 + n_cam_post_fisico)
    df["N_CAM_POST_FISICO"] = n_cam_post_fisico
    df["BRECHA_REMANENTE"] = (df["INDICE_100"] * factor_post).round(2)
    return df

df = calcular_brecha_remanente(df, asignacion)
(df[["OBJECTID", "Nombre_Bar", "Admin_Zona", "BRECHA", "N_CAM_POST_FISICO", "BRECHA_REMANENTE"]]
    .sort_values("BRECHA_REMANENTE", ascending=False).head(10))


,OBJECTID,Nombre_Bar,Admin_Zona,BRECHA,N_CAM_POST_FISICO,BRECHA_REMANENTE
5400,5401,LA FLORESTA (MARISCAL SUCRE),LA MARISCAL,73.935133,3,18.48
5446,5447,LA FLORESTA (MARISCAL SUCRE),LA MARISCAL,49.095438,3,12.27
4254,4255,CLEMENTE SÁNCHEZ DE ORELLANA,QUITUMBE,45.377689,3,11.34
4962,4963,SOLANDA SECTOR - 4,ELOY ALFARO,17.346276,7,10.84
4968,4969,BARRIO UNIÓN Y JUSTICIA,ELOY ALFARO,26.551185,4,10.62
14403,14404,SIERRA HERMOSA,CALDERÓN,40.854073,3,10.21
4229,4230,NUEVA AURORA,QUITUMBE,14.285714,9,10.00
13990,13991,LA FLORESTA (CARCELÉN),LA DELICIA,39.726095,3,9.93
4331,4332,SOLANDA SECTOR - 3,ELOY ALFARO,15.234549,6,8.71
4964,4965,FRENTE POPULAR,ELOY ALFARO,21.118153,4,8.45


## 6. Cobertura de fenómenos prioritarios, antes/después (§4.2)

In [9]:
def cobertura_por_fenomeno(df, componentes=COMPONENTES_INDICE):
    filas = []
    for var in componentes:
        sub = df[df[var] > 0]
        antes = (sub["N_CAM"] > 0).mean() * 100
        despues = (sub["N_CAM_POST"] > 0).mean() * 100
        filas.append((var, len(sub), round(antes, 1), round(despues, 1)))
    filas.append((
        "Grilla completa", len(df),
        round((df["N_CAM"] > 0).mean() * 100, 2),
        round((df["N_CAM_POST"] > 0).mean() * 100, 2),
    ))
    return pd.DataFrame(filas, columns=["Indicador", "Hexágonos con incidente", "Cobertura antes (%)", "Cobertura después (%)"])

cobertura = cobertura_por_fenomeno(df)
cobertura


,Indicador,Hexágonos con incidente,Cobertura antes (%),Cobertura después (%)
0,HI,191,33.50,100.00
1,ROBOS,1681,22.40,47.20
2,LIBADOR,1599,22.70,48.70
3,Grilla completa,18640,2.23,4.66


## 7. Cruce con espacio público y equipamientos institucionales (§4.5)

Generaliza la lógica aplicada primero a PARQUE: para cada variable de presencia (equipamiento urbano o programa institucional), compara la cobertura de cámaras antes/después dentro de ese universo. Incluye `PARQUE_15`, que identifica los 15 parques metropolitanos emblemáticos del DMQ dentro del universo más amplio de `PARQUE`.


In [10]:
VARIABLES_CRUCE = [
    "PARQUE", "PARQUE_15", "MERCADO", "UEDU", "P_BUS", "P_METRO", "SENDERO",
    "UPC", "PMO_CH", "PMO_MARIS", "UNIVERSID", "POLIG_ALT", "ZONA_0", "NZONA_0",
]

def cruce_variable(df, asignacion, variable):
    """Misma lógica que el cruce con PARQUE, aplicada a cualquier variable
    de presencia (>0 = pertenece al universo)."""
    universo_ids = set(df[df[variable] > 0]["OBJECTID"])
    cam_actual_ids = set(df[df["N_CAM"] > 0]["OBJECTID"])
    asign_ids = set(asignacion["OBJECTID"])
    post_ids = cam_actual_ids | asign_ids
    n = len(universo_ids)
    if n == 0:
        return None
    return {
        "Variable": variable,
        "N hexágonos": n,
        "% de la grilla": round(n / len(df) * 100, 2),
        "Cámaras antes": len(cam_actual_ids & universo_ids),
        "% cobertura antes": round(len(cam_actual_ids & universo_ids) / n * 100, 1),
        "Puntos de conexión asignados": len(asign_ids & universo_ids),
        "% de los 600 puntos": round(len(asign_ids & universo_ids) / len(asign_ids) * 100, 1),
        "% cobertura después": round(len(post_ids & universo_ids) / n * 100, 1),
    }

def cruce_todas_las_variables(df, asignacion, variables=VARIABLES_CRUCE):
    filas = [cruce_variable(df, asignacion, v) for v in variables]
    filas = [f for f in filas if f is not None]
    return pd.DataFrame(filas).sort_values("N hexágonos", ascending=False)

cruce_vars = cruce_todas_las_variables(df, asignacion)
cruce_vars


,Variable,N hexágonos,% de la grilla,Cámaras antes,% cobertura antes,Puntos de conexión asignados,% de los 600 puntos,% cobertura después
0,PARQUE,1235,6.63,300,24.3,442,73.7,50.6
4,P_BUS,1183,6.35,306,25.9,475,79.2,55.0
3,UEDU,911,4.89,254,27.9,375,62.5,55.8
1,PARQUE_15,287,1.54,49,17.1,51,8.5,31.0
7,UPC,269,1.44,107,39.8,161,26.8,77.0
6,SENDERO,72,0.39,47,65.3,40,6.7,87.5
10,UNIVERSID,64,0.34,23,35.9,32,5.3,62.5
11,POLIG_ALT,39,0.21,27,69.2,30,5.0,92.3
2,MERCADO,36,0.19,26,72.2,20,3.3,97.2
9,PMO_MARIS,16,0.09,9,56.2,12,2.0,100.0


## 7b. Desglose por parque metropolitano emblemático

El 31,0% agregado de cobertura en PARQUE_15 esconde una variación interna amplia: desde parques que llegan al 100% hasta otros en 0%. Esta celda desagrega por `Nm_15PARQ`, el nombre de cada uno de los 15 parques.


In [11]:
def cruce_parques_emblematicos(df, asignacion):
    """Mismo cruce que cruce_variable, pero desagregado por cada uno de
    los 15 parques metropolitanos emblemáticos (Nm_15PARQ)."""
    cam_actual_ids = set(df[df["N_CAM"] > 0]["OBJECTID"])
    asign_ids = set(asignacion["OBJECTID"])
    post_ids = cam_actual_ids | asign_ids

    sub = df[df["PARQUE_15"] > 0]
    filas = []
    for nombre, grp in sub.groupby("Nm_15PARQ"):
        ids = set(grp["OBJECTID"])
        n = len(ids)
        filas.append({
            "Parque": nombre,
            "N hexágonos": n,
            "Cámaras antes": len(cam_actual_ids & ids),
            "% cobertura antes": round(len(cam_actual_ids & ids) / n * 100, 1),
            "Puntos de conexión asignados": len(asign_ids & ids),
            "% cobertura después": round(len(post_ids & ids) / n * 100, 1),
        })
    return pd.DataFrame(filas).sort_values("N hexágonos", ascending=False)

parques_15 = cruce_parques_emblematicos(df, asignacion)
parques_15


,Parque,N hexágonos,Cámaras antes,% cobertura antes,Puntos de conexión asignados,% cobertura después
1,CHAQUIÑAN,129,12,9.3,14,20.2
14,Metropolitano del Sur,42,2,4.8,4,11.9
8,METROPOLITANO NORTE GUANGUILTAGUA,39,12,30.8,9,51.3
9,Metropolitano Chilibulo,28,2,7.1,1,7.1
0,BICENTENARIO,13,3,23.1,8,84.6
6,LA CAROLINA,9,7,77.8,8,100.0
13,Metropolitano La Armenia,6,0,0.0,0,0.0
12,Metropolitano Fundeporte (Carollo),4,2,50.0,1,75.0
7,Las Cuadras,4,2,50.0,2,75.0
11,Metropolitano Equinoccial,3,0,0.0,1,33.3


## 7b. Validación externa: Índice de Convivencia Ciudadana (ICC)

El ICC mide incivilidades (libadores, riñas, escándalos en vía pública) en cuatro categorías (Bajo/Moderado/Alto/Muy Alto), o "0" sin clasificar. **No se usó** para construir el índice de intensidad de inseguridad ni la brecha: sirve como validación externa e independiente.


In [12]:
ICC_ORDEN = ["0", "Bajo", "Moderado", "Alto", "Muy Alto"]

def validar_icc(df, asignacion):
    cam_actual_ids = set(df[df["N_CAM"] > 0]["OBJECTID"])
    asign_ids = set(asignacion["OBJECTID"])
    post_ids = cam_actual_ids | asign_ids

    df = df.copy()
    df["ICC"] = pd.Categorical(df["ICC"].astype(str), categories=ICC_ORDEN, ordered=True)

    filas = []
    for cat in ICC_ORDEN:
        ids = set(df[df["ICC"] == cat]["OBJECTID"])
        n = len(ids)
        if n == 0:
            continue
        filas.append({
            "Categoría ICC": cat,
            "N hexágonos": n,
            "Cámaras antes": len(cam_actual_ids & ids),
            "% cobertura antes": round(len(cam_actual_ids & ids) / n * 100, 1),
            "Puntos de conexión asignados": len(asign_ids & ids),
            "% cobertura después": round(len(post_ids & ids) / n * 100, 1),
        })
    return pd.DataFrame(filas)

icc = validar_icc(df, asignacion)
icc


,Categoría ICC,N hexágonos,Cámaras antes,% cobertura antes,Puntos de conexión asignados,% cobertura después
0,0,16565,35,0.2,13,0.3
1,Bajo,1919,282,14.7,463,34.8
2,Moderado,52,28,53.8,40,98.1
3,Alto,52,30,57.7,43,100.0
4,Muy Alto,52,40,76.9,41,98.1


## 7c. Brecha remanente agregada — lo que faltaría para el 100% (§4.7)

Cuántos puntos de conexión (y cámaras físicas equivalentes) harían falta, además de los 600 ya asignados, para que cada universo llegue al 100% de cobertura.


In [13]:
def brecha_remanente_agregada(df, asignacion):
    cam_actual_ids = set(df[df["N_CAM"] > 0]["OBJECTID"])
    asign_ids = set(asignacion["OBJECTID"])
    post_ids = cam_actual_ids | asign_ids

    universos = {
        "Homicidios": df["HI"] > 0,
        "Robos": df["ROBOS"] > 0,
        "Libadores": df["LIBADOR"] > 0,
        "Parques y plazas": df["PARQUE"] > 0,
        "Paradas de bus": df["P_BUS"] > 0,
        "Unidades educativas": df["UEDU"] > 0,
        "UPC": df["UPC"] > 0,
    }
    filas = []
    for nombre, mask in universos.items():
        ids = set(df[mask]["OBJECTID"])
        cubiertos = ids & post_ids
        faltan = ids - post_ids
        filas.append({
            "Universo": nombre,
            "Cobertura actual": round(len(cubiertos) / len(ids) * 100, 1),
            "Puntos que faltan": len(faltan),
            "Cámaras que faltan (×3)": len(faltan) * CAMARAS_POR_PUNTO,
        })

    icc_ids = set(df[df["ICC"].astype(str).isin(["Bajo", "Moderado", "Alto", "Muy Alto"])]["OBJECTID"])
    faltan_icc = icc_ids - post_ids
    filas.append({
        "Universo": "ICC clasificado (4 categorías)",
        "Cobertura actual": round(len(icc_ids & post_ids) / len(icc_ids) * 100, 1),
        "Puntos que faltan": len(faltan_icc),
        "Cámaras que faltan (×3)": len(faltan_icc) * CAMARAS_POR_PUNTO,
    })

    total_ids = set(df["OBJECTID"])
    faltan_total = total_ids - post_ids
    filas.append({
        "Universo": "Grilla completa (18.640 hex.)",
        "Cobertura actual": round(len(post_ids) / len(df) * 100, 1),
        "Puntos que faltan": len(faltan_total),
        "Cámaras que faltan (×3)": len(faltan_total) * CAMARAS_POR_PUNTO,
    })
    return pd.DataFrame(filas)

brecha_remanente_agregada(df, asignacion)


,Universo,Cobertura actual,Puntos que faltan,Cámaras que faltan (×3)
0,Homicidios,100.0,0,0
1,Robos,47.2,888,2664
2,Libadores,48.7,820,2460
3,Parques y plazas,50.6,610,1830
4,Paradas de bus,55.0,532,1596
5,Unidades educativas,55.8,403,1209
6,UPC,77.0,62,186
7,ICC clasificado (4 categorías),39.6,1253,3759
8,Grilla completa (18.640 hex.),4.7,17771,53313


## 8. Resumen por zona administrativa (§4.3, insumo del visualizador HTML)

In [14]:
def resumen_zonas(df, asignacion, cuota, pct):
    filas = []
    puntos_nuevos_zona = asignacion.groupby("Admin_Zona").size()
    for zona, grp in df.groupby("Admin_Zona"):
        n_cam = grp["N_CAM"].sum()
        n_puntos = int(puntos_nuevos_zona.get(zona, 0))
        n_camaras_nuevas = n_puntos * CAMARAS_POR_PUNTO
        filas.append({
            "z": zona,
            "hex": len(grp),
            "camEp": int(grp["CAM_EP"].sum()),
            "camEcu": int(grp["CAM_ECU"].sum()),
            "camActual": int(n_cam),
            "hexConCam": int((grp["N_CAM"] > 0).sum()),
            "pctHexConCam": round((grp["N_CAM"] > 0).mean() * 100, 1),
            "hi": int(grp["HI"].sum()),
            "robos": int(grp["ROBOS"].sum()),
            "libador": int(grp["LIBADOR"].sum()),
            "pctBrecha": round(pct.get(zona, 0), 1),
            "puntosNuevos": n_puntos,
            "camarasNuevas": n_camaras_nuevas,
            "camTotalPost": int(n_cam) + n_camaras_nuevas,
        })
    return sorted(filas, key=lambda r: -r["pctBrecha"])

zonas = resumen_zonas(df, asignacion, cuota, pct)
pd.DataFrame(zonas)


,z,hex,camEp,camEcu,camActual,hexConCam,pctHexConCam,hi,robos,libador,pctBrecha,puntosNuevos,camarasNuevas,camTotalPost
0,EUGENIO ESPEJO,2780,101,129,230,81,2.9,28,3832,5611,18.9,113,339,569
1,LA DELICIA,915,38,79,117,41,4.5,44,1713,5551,16.0,96,288,405
2,ELOY ALFARO,2620,133,114,247,67,2.6,50,1953,9519,14.2,85,255,502
3,QUITUMBE,436,94,55,149,55,12.6,55,1735,4427,13.8,83,249,398
4,CALDERÓN,373,34,48,82,33,8.8,17,1122,2387,8.3,50,150,232
5,TUMBACO,2806,60,42,102,39,1.4,24,904,2382,8.3,50,150,252
6,MANUELA SÁENZ,229,231,136,367,55,24.0,27,1498,6393,6.9,41,123,490
7,LOS CHILLOS,2965,16,20,36,18,0.6,8,942,1896,6.7,40,120,156
8,LA MARISCAL,72,93,62,155,23,31.9,12,1320,4029,6.0,36,108,263
9,CHOCO ANDINO,5444,0,4,4,3,0.1,6,26,282,1.0,6,18,22


## 9. Georreferenciación y capas del mapa (opcional)

El geojson de origen no trae CRS declarado. Se calibra por transformación de similitud (escala + rotación + traslación), ajustada por mínimos cuadrados con los centroides de las 10 zonas administrativas. Esta celda solo corre si el geojson está presente en el directorio de trabajo.


In [15]:
TRANSFORM = {"a": 8.90314712e-06, "b": -1.09014735e-08, "c": -83.0610751, "d": -89.0282066}

def proyectada_a_wgs84(x, y, t=TRANSFORM):
    lon = t["a"] * x - t["b"] * y + t["c"]
    lat = t["b"] * x + t["a"] * y + t["d"]
    return lon, lat

def construir_capas_mapa(ruta_geojson, df_indice, asignacion):
    """Genera las tres capas del visualizador: hexágonos con brecha>0,
    cámaras actuales (centroide) y puntos de conexión nuevos (centroide)."""
    idx = df_indice.set_index("OBJECTID")
    asign_ids = set(asignacion["OBJECTID"])

    with open(ruta_geojson, encoding="utf-8") as f:
        hg = json.load(f)

    hex_brecha, cam_actuales, cam_nuevas = [], [], []
    for feat in hg["features"]:
        oid = int(feat["properties"]["OBJECTID"])
        if oid not in idx.index:
            continue
        row = idx.loc[oid]
        coords = feat["geometry"]["coordinates"][0][0]
        ring_wgs = [proyectada_a_wgs84(x, y) for x, y in coords]
        cx = sum(p[0] for p in ring_wgs) / len(ring_wgs)
        cy = sum(p[1] for p in ring_wgs) / len(ring_wgs)

        if row["BRECHA"] > 0:
            hex_brecha.append({
                "id": oid, "r": [[round(lo, 6), round(la, 6)] for lo, la in ring_wgs],
                "br": round(row["BRECHA"], 2), "ix": round(row["INDICE_100"], 2),
                "hi": int(row["HI"]), "ro": int(row["ROBOS"]), "li": int(row["LIBADOR"]),
                "b": row["Nombre_Bar"], "z": row["Admin_Zona"], "cam": int(row["N_CAM"]),
            })
        if row["N_CAM"] > 0:
            cam_actuales.append({
                "id": oid, "lat": round(cy, 6), "lon": round(cx, 6),
                "ep": int(row["CAM_EP"]), "ecu": int(row["CAM_ECU"]),
                "b": row["Nombre_Bar"], "z": row["Admin_Zona"],
            })
        if oid in asign_ids:
            cam_nuevas.append({
                "id": oid, "lat": round(cy, 6), "lon": round(cx, 6),
                "br": round(row["BRECHA"], 2), "hi": int(row["HI"]),
                "ro": int(row["ROBOS"]), "li": int(row["LIBADOR"]),
                "b": row["Nombre_Bar"], "z": row["Admin_Zona"],
            })
    return hex_brecha, cam_actuales, cam_nuevas

import os
RUTA_GEOJSON = "hex_300m_incidentes_v2_AZ.geojson"
if os.path.exists(RUTA_GEOJSON):
    hex_brecha, cam_actuales, cam_nuevas = construir_capas_mapa(RUTA_GEOJSON, df, asignacion)
    print(f"hexágonos con brecha>0: {len(hex_brecha)} | cámaras actuales: {len(cam_actuales)} | puntos nuevos: {len(cam_nuevas)}")
else:
    print(f"'{RUTA_GEOJSON}' no encontrado en el directorio de trabajo — se omite esta celda.")


hexágonos con brecha>0: 2033 | cámaras actuales: 415 | puntos nuevos: 600


## 10. Exportar resultados

In [16]:
cols_indice = ["OBJECTID", "Nombre_Bar", "Parroquia", "Admin_Zona", "AÑO",
               "LIBADOR", "ESCAN", "VCSSF", "ROBOS", "HI", "CAM_EP", "CAM_ECU",
               "PARQUE", "MERCADO", "UEDU", "P_BUS", "P_METRO", "SENDERO", "UPC",
               "PMO_CH", "PMO_MARIS", "UNIVERSID", "POLIG_ALT", "ICC", "ZONA_0",
               "NZONA_0", "PARQUE_15", "Nm_15PARQ",
               "N_CAM", "INDICE_100", "BRECHA", "N_CAM_POST_FISICO", "BRECHA_REMANENTE"]
df[cols_indice].to_csv("hex_indice.csv", index=False)

cols_asign = ["OBJECTID", "Nombre_Bar", "Parroquia", "Admin_Zona", "HI",
              "ROBOS", "LIBADOR", "N_CAM", "INDICE_100", "BRECHA"]
asignacion[cols_asign].to_csv("asignacion_600_puntos_conexion.csv", index=False)

validacion.to_csv("validacion_correlacion.csv", index=False)
cruce_vars.to_csv("cruce_espacio_publico_equipamientos.csv", index=False)
parques_15.to_csv("cruce_parques_emblematicos.csv", index=False)
icc.to_csv("validacion_icc.csv", index=False)

with open("zonas_stats.json", "w", encoding="utf-8") as f:
    json.dump(zonas, f, ensure_ascii=False, indent=1)

print("Archivos generados: hex_indice.csv, asignacion_600_puntos_conexion.csv,")
print("validacion_correlacion.csv, cruce_espacio_publico_equipamientos.csv,")
print("cruce_parques_emblematicos.csv,")
print("validacion_icc.csv, zonas_stats.json")
print()
print(f"Puntos de conexión nuevos: {N_PUNTOS_NUEVOS} (= {N_PUNTOS_NUEVOS * CAMARAS_POR_PUNTO} cámaras físicas, {CAMARAS_POR_PUNTO} por punto)")


Archivos generados: hex_indice.csv, asignacion_600_puntos_conexion.csv,
validacion_correlacion.csv, cruce_espacio_publico_equipamientos.csv,
cruce_parques_emblematicos.csv,
validacion_icc.csv, zonas_stats.json

Puntos de conexión nuevos: 600 (= 1800 cámaras físicas, 3 por punto)
